# scikit-learn 现代建模流程

这个 notebook 用 Titanic 示例数据演示当前更推荐的机器学习工作流：把数据拆分、缺失值处理、类别编码、数值缩放、模型训练和评估放进可复用的 `Pipeline`。


## 为什么用 Pipeline

直接在完整数据上做预处理再切分，容易把验证集信息泄漏进训练过程。`Pipeline` 和 `ColumnTransformer` 可以把预处理步骤绑定到模型上，让交叉验证和线上推理都走同一套流程。


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("../../resources/datasets/titanic.csv")

titanic = pd.read_csv(DATA_PATH)
titanic.head()


## 选择特征和目标

这里保留少量结构化特征，重点展示流程。真实项目中应结合业务含义、数据质量和偏差风险重新做特征选择。


In [ ]:
target = "Survived"
numeric_features = ["Pclass", "Age", "Fare"]
categorical_features = ["Sex", "Embarked"]
features = numeric_features + categorical_features

X = titanic[features]
y = titanic[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


## 构建预处理和模型管道

数值列使用中位数补缺失值并标准化；类别列使用众数补缺失值并做 one-hot 编码。`handle_unknown="ignore"` 可以避免预测阶段遇到新类别时报错。


In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(max_iter=1_000)),
    ]
)

model


## 交叉验证

交叉验证直接作用在完整管道上，避免每折训练和验证之间的数据泄漏。


In [ ]:
scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=5,
    scoring=["accuracy", "roc_auc"],
)

pd.DataFrame(scores).agg(["mean", "std"]).T


## 在测试集上评估

测试集只在最终评估阶段使用。模型训练仍然调用同一个 `Pipeline`，预测时会自动执行相同的预处理步骤。


In [ ]:
model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
print(f"ROC AUC: {roc_auc_score(y_test, probabilities):.3f}")
print(classification_report(y_test, predictions))


## 下一步

可以继续把 `classifier` 替换为 `RandomForestClassifier`、`HistGradientBoostingClassifier` 或其他估计器；只要保持同一个 `Pipeline` 接口，训练、交叉验证和推理代码都不需要大改。
